<a href="https://colab.research.google.com/github/jjyjung/ai/blob/gh-pages/rnn/pytorch_RNN_IMDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **IMDB (Internet Movie DataBase ) – movie reviews sentiment classification**
* https://www.imdb.com/interfaces/
* https://keras.io/datasets/#imdb-movie-reviews-sentiment-classification
* Binary classification: Dataset of 25,000 movies reviews from IMDB, labeled by sentiment (positive[1]/negative[0]).
* Each review is encoded as a sequence of word indexes (integers).
* For convenience, words are indexed by overall frequency in the dataset.
* This allows for quick filtering operations such as: “only consider the top 10,000 most common words, but eliminate the top 20 most common words”.
 - e.g.  imdb.load_data(num_words=10000, skip_top=20)
* As a convention, "0" does not stand for a specific word, but instead is used to encode any unknown word.

PyTorch 기반 학습에 필요한 라이브러리와 학습, 평가, 시각화 함수를 불러옵니다.

In [ ]:
import copy
import json
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, Subset


def pad_reviews(sequences, maxlen):
    """review가 maxlen(예.200)보다 짧으면 앞쪽을 0으로 채우고, 길면 앞부분을 자릅니다."""
    padded = np.zeros((len(sequences), maxlen), dtype=np.int64)
    for i, sequence in enumerate(sequences):
        sequence = np.asarray(sequence[-maxlen:], dtype=np.int64)
        if len(sequence):
            padded[i, -len(sequence):] = sequence
    return padded


def run_epoch(model, loader, criterion, device, optimizer=None):
    is_training = optimizer is not None
    model.train(is_training)
    total_loss, total_correct, total_count = 0.0, 0, 0

    for inputs, labels in loader:
        inputs = inputs.to(device)
        labels = labels.to(device).float()
        if is_training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(is_training):
            logits = model(inputs)
            loss = criterion(logits, labels)
            if is_training:
                loss.backward()
                optimizer.step()
        # 평균 loss 계산
        count = labels.size(0)
        total_loss += loss.item() * count
        # 평균 accuracy 계산
        predictions = (logits >= 0).float()
        total_correct += (predictions == labels).sum().item()
        total_count += count

    return total_loss / total_count, total_correct / total_count


def fit_model(model, train_loader, val_loader, device, learning_rate=0.001,
              epochs=100, patience=10):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    history = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}
    best_val_loss = float('inf')
    best_weights = copy.deepcopy(model.state_dict())
    epochs_without_improvement = 0

    for epoch in range(epochs):
        train_loss, train_accuracy = run_epoch(
            model, train_loader, criterion, device, optimizer
        )
        val_loss, val_accuracy = run_epoch(model, val_loader, criterion, device)
        history['loss'].append(train_loss)
        history['accuracy'].append(train_accuracy)
        history['val_loss'].append(val_loss)
        history['val_accuracy'].append(val_accuracy)
        print(f"Epoch {epoch + 1:03d}/{epochs} - loss: {train_loss:.4f} - "
              f"accuracy: {train_accuracy:.4f} - val_loss: {val_loss:.4f} - "
              f"val_accuracy: {val_accuracy:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"검증 손실 개선이 없어 {epoch + 1} 에폭에서 조기 종료합니다.")
                break

    model.load_state_dict(best_weights)
    return history


def show_model_summary(model):
    print(model)
    total = sum(parameter.numel() for parameter in model.parameters())
    trainable = sum(parameter.numel() for parameter in model.parameters()
                    if parameter.requires_grad)
    trainable_ratio = trainable / total * 100

    print(f"전체 파라미터 수: {total:,}")
    print(f"학습 가능한 파라미터 수: {trainable:,}")
    print(f"학습 가능한 파라미터 비율: {trainable_ratio:.2f}%")


def plot_history(history):
    fig, loss_ax = plt.subplots(figsize=(8, 5))
    accuracy_ax = loss_ax.twinx()

    loss_ax.plot(history['loss'], 'y', label='train loss')
    loss_ax.plot(history['val_loss'], 'r', label='validation loss')
    accuracy_ax.plot(history['accuracy'], 'b', label='train accuracy')
    accuracy_ax.plot(history['val_accuracy'], 'g', label='validation accuracy')

    loss_ax.set_xlabel('Epoch')
    loss_ax.set_ylabel('Loss')
    accuracy_ax.set_ylabel('Accuracy')
    loss_ax.legend(loc='upper left')
    accuracy_ax.legend(loc='lower left')
    loss_ax.grid(True)
    plt.show()

IMDB 정수 인코딩 데이터를 내려받아 Keras 예제와 같은 상위 단어 제한, 상위 빈도 단어 제외, 200개 길이 앞쪽 패딩을 적용합니다.

공식 학습 세트 안에서 클래스별 20%를 검증 세트로 분리하고, 공식 테스트 세트는 마지막 평가용으로 둡니다.

In [ ]:
# Keras IMDB 공개 정수 인코딩 데이터 파일을 받아옵니다.
max_features = 10000
maxlen = 200
skip_top = 20
embedding_dim = 100
batch_size = 100

cache_dir = Path('./data')
cache_dir.mkdir(parents=True, exist_ok=True)
npz_path = cache_dir / 'imdb.npz'
data_url = 'https://storage.googleapis.com/tensorflow/tf-keras-datasets/imdb.npz'
if not npz_path.exists():
    print('IMDB 데이터 다운로드 중...')
    urllib.request.urlretrieve(data_url, npz_path)

# 신뢰하는 공식 데이터 파일에 대해서만 load 실행 필요 (보안 문제)
with np.load(npz_path, allow_pickle=True) as data:
    input_train = data['x_train'].tolist()
    y_train = data['y_train'].astype(np.int64)
    input_test = data['x_test'].tolist()
    y_test = data['y_test'].astype(np.int64)

def apply_keras_imdb_indexing(sequences):
    processed = []
    for sequence in sequences:
        # index_from=3, start_char=1, oov_char=2와 같은 처리
        indexed = [1] + [int(word) + 3 for word in sequence]
        indexed = [word if skip_top <= word < max_features else 2
                   for word in indexed]
        processed.append(indexed)
    return processed

input_train = apply_keras_imdb_indexing(input_train)
input_test = apply_keras_imdb_indexing(input_test)
X_train = pad_reviews(input_train, maxlen)
X_test = pad_reviews(input_test, maxlen)

# 클래스별로 학습 데이터의 20%를 검증용으로 분리합니다.
train_indices, val_indices = [], []
val_rng = np.random.default_rng(42)
for label in (0, 1):
    class_indices = np.flatnonzero(y_train == label)
    val_rng.shuffle(class_indices) # 각 클래스에서 무작위로 뽑기 위해 필요

    val_count = int(round(len(class_indices) * 0.2))
    val_indices.extend(class_indices[:val_count].tolist())
    train_indices.extend(class_indices[val_count:].tolist())

train_full = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
train_dataset = Subset(train_full, train_indices)
val_dataset = Subset(train_full, val_indices)
test_dataset = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test))

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"사용 장치: {device}")
print(f"train: {len(train_dataset):,}, validation: {len(val_dataset):,}, "
      f"test: {len(test_dataset):,}")

전처리한 입력 행렬과 정답 레이블의 크기 및 단어 인덱스 범위를 확인합니다.

In [ ]:
print('X_train:', X_train.shape)
print('X_test:', X_test.shape)
print('단어 인덱스 범위:', X_train.max(), X_train.min())
print('y_train:', y_train, y_train.shape)
print('y_test:', y_test, y_test.shape)

In [ ]:
print(f"train: {len(train_dataset):,}, validation: {len(val_dataset):,}, test: {len(test_dataset):,}")
# NumPy의 bincount 함수는 0부터 시작하여 각 정수 값(0, 1, 등)이 등장한 빈도수를 세어 배열로 반환
print("클래스별 레이블 수 (train):", np.bincount(y_train[train_indices], minlength=2))
print("클래스별 레이블 수 (validation):", np.bincount(y_train[val_indices], minlength=2))
print("클래스별 레이블 수 (test):", np.bincount(y_test, minlength=2))

패딩 전 리뷰 시퀀스와 길이, 패딩 후 고정 길이 시퀀스를 비교합니다.

In [ ]:
print(input_train[3:5])
print(len(input_train[3]), len(input_train[4]))
print(X_train[3:5])
print(len(X_train[3]), len(X_train[4]))

IMDB 단어 사전 JSON을 내려받아 단어별 인덱스를 역방향으로 바꿉니다. 실제 데이터의 단어 인덱스는 예약 토큰을 위해 3을 더한 값입니다.

In [ ]:
word_index_path = cache_dir / 'imdb_word_index.json'
word_index_url = 'https://storage.googleapis.com/tensorflow/tf-keras-datasets/imdb_word_index.json'
if not word_index_path.exists():
    urllib.request.urlretrieve(word_index_url, word_index_path)
with open(word_index_path, encoding='utf-8') as file:
    word_to_index = json.load(file)

index_to_word = {value + 3: word for word, value in word_to_index.items()}
print('단어 사전 항목 수:', len(word_to_index))
print(list(index_to_word.items())[:5])

인덱스 순서로 정렬해 IMDB 단어 사전에서 가장 앞쪽에 있는 단어들을 확인합니다.

In [ ]:
sorted_index_to_word = dict(sorted(index_to_word.items()))
list(sorted_index_to_word.items())[:20]

패딩, 문장 시작, 미등록 단어, 예약 인덱스에 해당하는 특수 토큰을 역방향 사전에 추가합니다.

In [ ]:
for index, token in enumerate(("<PAD>", "<START>", "<UNK>", "<UNUSED>")):
    index_to_word[index] = token

패딩된 리뷰의 정수 인덱스를 실제 단어와 특수 토큰으로 복원해 몇 개의 리뷰를 출력해 봅니다.

In [ ]:
for review in X_train[:5]:
    print(' '.join(index_to_word.get(int(index), '<UNK>') for index in review))

### Single LSTM

단어 임베딩 뒤 단일 LSTM 층과 로짓 출력층을 연결합니다.

세 모델은 모두 Adam과 BCEWithLogitsLoss로 학습하며, validation loss로 조기 종료와 최적 가중치를 결정합니다.

test 데이터는 마지막 모델 선택 후 최종 평가에만 사용합니다.

In [ ]:
class SingleLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim=100, hidden_size=32):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_size, batch_first=True)
        self.classifier = nn.Linear(hidden_size, 1)

    def forward(self, tokens):
        embedded = self.embedding(tokens)
        _, (hidden, _) = self.lstm(embedded)
        return self.classifier(hidden[-1]).squeeze(1)

model_lstm = SingleLSTM(max_features, embedding_dim, hidden_size=32).to(device)
hist_lstm = fit_model(model_lstm, train_loader, val_loader, device)

단일 LSTM의 계층 구조와 전체 파라미터 수를 확인합니다.

In [ ]:
show_model_summary(model_lstm)

단일 LSTM의 학습, 검증 손실과 정확도를 에폭별 그래프로 표시합니다.

In [ ]:
plot_history(hist_lstm)

테스트 데이터에 대하여 loss 및 accuracy를 확인합니다.

In [ ]:
test_loss, test_accuracy = run_epoch(
    model_lstm, test_loader, nn.BCEWithLogitsLoss(), device
)
print(f"test loss: {test_loss:.4f}")
print(f"test accuracy: {test_accuracy:.4f}")

### Stacked LSTM

두 개의 LSTM 층을 쌓은 모델을 정의하고 학습합니다.

PyTorch nn.LSTM에는 Keras recurrent_dropout 옵션이 없어 입력 및 층 사이에 dropout을 적용합니다.

validation loss로 최적 가중치와 조기 종료 시점을 정하며, optimizer와 손실 함수는 다른 두 모델과 동일하게 사용합니다.

In [ ]:
class StackedLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim=100,
                 first_hidden=128, second_hidden=64, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.input_dropout = nn.Dropout(dropout)
        self.lstm1 = nn.LSTM(embedding_dim, first_hidden, batch_first=True)
        self.inter_layer_dropout = nn.Dropout(dropout)
        self.lstm2 = nn.LSTM(first_hidden, second_hidden, batch_first=True)
        self.classifier = nn.Linear(second_hidden, 1)

    def forward(self, tokens):
        embedded = self.input_dropout(self.embedding(tokens))
        sequence, _ = self.lstm1(embedded)
        sequence = self.inter_layer_dropout(sequence)
        _, (hidden, _) = self.lstm2(sequence)
        return self.classifier(hidden[-1]).squeeze(1)

model_stacked = StackedLSTM(max_features, embedding_dim,
                            first_hidden=128, second_hidden=64, dropout=0.3).to(device)
hist_stacked = fit_model(model_stacked, train_loader, val_loader, device)

Stacked LSTM의 계층 구조와 파라미터 수를 확인합니다.

In [ ]:
show_model_summary(model_stacked)

Stacked LSTM의 학습, 검증 손실과 정확도를 에폭별 그래프로 표시합니다.

In [ ]:
plot_history(hist_stacked)

테스트 데이터에 대하여 loss 및 accuracy를 확인합니다.

In [ ]:
test_loss, test_accuracy = run_epoch(
    model_stacked, test_loader, nn.BCEWithLogitsLoss(), device
)
print(f"test loss: {test_loss:.4f}")
print(f"test accuracy: {test_accuracy:.4f}")

### Bidirectional LSTM

양방향 LSTM의 마지막 은닉 상태를 이어 붙여 분류합니다.

PyTorch nn.LSTM에는 Keras recurrent_dropout과 같은 옵션이 없으므로 입력과 dense 층에 dropout을 적용합니다.

In [ ]:
class BidirectionalLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim=100, hidden_size=100,
                 dense_size=64, input_dropout=0.4, dense_dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.input_dropout = nn.Dropout(input_dropout)
        self.lstm = nn.LSTM(embedding_dim, hidden_size, batch_first=True,
                            bidirectional=True)
        # PyTorch nn.LSTM에는 Keras recurrent_dropout과 동일한 옵션이 없습니다.
        self.dense = nn.Linear(hidden_size * 2, dense_size)
        self.dropout = nn.Dropout(dense_dropout)
        self.classifier = nn.Linear(dense_size, 1)

    def forward(self, tokens):
        embedded = self.input_dropout(self.embedding(tokens))
        _, (hidden, _) = self.lstm(embedded)
        bidirectional_hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        features = torch.relu(self.dense(bidirectional_hidden))
        features = self.dropout(features)
        return self.classifier(features).squeeze(1)

model_bilstm = BidirectionalLSTM(max_features, embedding_dim,
                                hidden_size=embedding_dim).to(device)
hist_bilstm = fit_model(model_bilstm, train_loader, val_loader, device)

양방향 LSTM의 네트워크 구조와 파라미터 수를 확인합니다.

In [ ]:
show_model_summary(model_bilstm)

양방향 LSTM의 학습, 검증 손실과 정확도를 그래프로 확인합니다.

In [ ]:
plot_history(hist_bilstm)

테스트 데이터에 대하여 loss 및 accuracy를 확인합니다.

In [ ]:
test_loss, test_accuracy = run_epoch(
    model_bilstm, test_loader, nn.BCEWithLogitsLoss(), device
)
print(f"test loss: {test_loss:.4f}")
print(f"test accuracy: {test_accuracy:.4f}")